# Terminal Interface

*Written by Grace Lo & Alina Wang*

This integrated terminal interface system combines SD card functionality with motor testing and CO₂ and methane sensor measurements, displayed on a serial monitor mimicking your computer's terminal or PowerShell command-line interface. It has a series of valid commands to navigate and alter files/directories on the SD card, as well as change the motor or data logging settings of the system. Like the [`flux_chamber.c`](../Flux_Chamber/Flux_Chamber.ipynb) system, it can also collect environmental data from both the [Sensirion SCD30](https://sensirion.com/products/catalog/SCD30) (CO₂, temperature, humidity) and the [Figaro NGM2611-E13](https://www.figarosensor.com/product/entry/ngm2611.html) (methane), and write this data with timestamps to a `.csv` file. 

All base motor, SD card and sensor functionality is handled by their respective libraries,  documented on the [Stepper Motor](../Stepper_Motor/Stepper_Motor.html), [CO₂ Sensor](../CO2/CO2.html), [Methane Sensor](../Methane/Methane.html) and [SD Card](../SD_Card/SD_Card.html) pages. The intent of this document is to show how these modules are integrated with a command-line interface for testing specific setting changes without the need to compile the code and reflash the Pico. *As such, this page will often refer to [Flux Chamber](../Flux_Chamber/Flux_Chamber.html) for various steps and mainly highlight the differences.*  

## Pin Connections

Breadboarding best practices should be followed. Use the shortest possible wires for maximum clock speed, which when using solderless protoboards is capped at ~20MHz due to pin capacitance. The pin connection for each components is already documented on their respective pages and the integrated version simply connects all the components together.

<img src="img/pin_connections.png" alt="Pin Connections" width="800"/>

## API

The serial monitor can be connected using the [Raspberry Pi Debug Probe](https://www.raspberrypi.com/products/debug-probe/) or the [Adafruit #954](https://www.adafruit.com/product/954) USB to TTL Serial cable, both of which can be connected via: 
- RX &rarr; UART0_TX (gpio 0)
- TX &rarr; UART0_RX (gpio 1)
- GND &rarr; GND

When the system resets, the driver prints SD card information to the serial terminal. Then you can issue the following commands:
- `test_motor <direction>`: run the motor up/down & stop it using the corresponding feedback switch
- `set_date <yyyy-mm-dd> <hh:mm:ss>`: set the RTC intialization of date/time
- `set_per <period>`: set the data logging period in ms (default set to 5 seconds)
- `write <filename>`: write data logging to a file
- `print <filename>`: print the content of the filename
- `rm <filename>`: delinks (deletes) a file
- `ls <directory>`: lists directory contents; default is current directory
- `cd <directory>`: changes current directory
- `mkdir <directory>`: new directory
- `help`: prints all valid commands

## Code

All code is in the [CornellFluxChamber](https://github.com/CornellFluxChamber/ChamberCode.git) Github repository. The file `terminal_interface.c` contains all core logic for sensor initialization, data logging, and command-line interactions. The code is organized in protothreads for modularity. Protothreads are extremely lightweight, stackless threads designed for memory-constrained systems; the library is written entirely as C macros by [Adam Dunkels](http://dunkels.com/adam/pt/publications.html).

### Header Files

The first lines of code in the C source file include header files. *Don't forget to link these in the CMakeLists.txt file!* All of the header files are the same as [`flux_chamber.c`](../Flux_Chamber/Flux_Chamber.html), except for the `pico/sleep.h` library that is not necessary for this implementation.

### `main()`

Refer to [`flux_chamber.c`](../Flux_Chamber/Flux_Chamber.html) for the core allocation and protohread setup prcoedures. 

### Protothread

#### Initializations

The protothread begins with variable declarations for processing serial input (`cmd`, `arg1`, `arg2`, `arg3`, `token`), date/time (`datetime_buf`, `datetime_str`), and frequency (`logging_frequency_ms`). In addition, the `datetime_t` struct is initialized to a default Sunday 01 January 00:00:00 2000 and the logging frequency is initialized to a default 5000 ms (or 5 seconds).

Then the SD card, CO₂ sensor, and ADC for the methane sensor are initialized in the same way as [`flux_chamber.c`](../Flux_Chamber/Flux_Chamber.html). 

#### Serial Terminal

The serial terminal runs indefinitely when the RP2040 is powered on. At the start of each loop, it waits for user input, then tokenizes `cmd arg1 arg2 arg3`. Each of the commands are checked in sequential if-statements and are described below:

The `test_motor` command uses `arg1` to determine the direction to run the motor and the proper switch to test for. When the motor moves up, it uses the limit switch to stop. When the motor moves down, it uses the float switch to stop. 

The `set_date` command uses `arg1` and `arg2` to set the date and time, respectively. The date is tokenized by delimiter `-` and the time is tokenized by delimiter `:`. The fields for the `datetime_t` struct are updated accordingly and reinitialized in the real time clock.

The `set_per` command uses `arg1` to set the period (in ms) for data logging. When the `write` command is issued, the period is used to put the system to sleep in between data logging entries.

The `write` command uses `arg1` as the filename to write to. If the file does not exist, a new one will be created. If the file already exists, new data will be appended to the file. The contents of the file are divided into two sections - header and data. The header includes the Pico ID so that each system can be identified uniquely when many flux chambers are deployed. It also defines each of the columns. At present, the columns are `DATETIME, TEMP, HUM, CO2, CH4`. Then the data is continuously written in a loop. As mentioned previously, the logging period is used to specify the sleep time between data logging entries.

The `rm` command uses `arg1` to define the file path for removing the file. 

The `ls` command can be used with or without `arg1`. `ls` alone will print all items in the current directory and `ls arg1` will print all items in the `arg1` directory. 

The `cd` command uses `arg1` to define the file path to navigate to. 

The `mkdir` command uses `arg1` to name the new directory that will be created in the current directory. 

An example of `setdate`, `setfreq`, `write`, and `print` used in the serial terminal and the corresponding CSV file is provided below:

<img src="img/serial_monitor.png" alt="Serial Terminal" width="400"> <img src="img/csv_file.png" alt="CSV File" width="400"/>